In [ ]:
import numpy as np
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

from copy import copy

In [ ]:
ini_dates = [2021070912, 2021071012, 2021071112, 2021071212]

In [ ]:
grid_data = {}

# Grid info for deterministic runs:
grid_data["grid_det"] = xr.open_dataset("./grids/icon_grid_0026_R03B07_G.nc")
grid_data["tri_det"] = Triangulation(np.rad2deg(grid_data["grid_det"]["clon"]), np.rad2deg(grid_data["grid_det"]["clat"]))
grid_data["area_det"] = grid_data["grid_det"]["cell_area"]

# Grid info for ensemble runs:
grid_data["grid_ens"] = xr.open_dataset("./grids/icon_grid_0028_R02B07_N02.nc")
grid_data["tri_ens"] = Triangulation(np.rad2deg(grid_data["grid_ens"]["clon"]), np.rad2deg(grid_data["grid_ens"]["clat"]))
grid_data["area_ens"] = grid_data["grid_ens"]["cell_area"]


# Define "study area"
x0, y0 = 3, 48
wx, wy = 7, 5.5

grid_data["cells_det"] = ((np.rad2deg(grid_data["grid_det"]["clon"]) >= x0) & (np.rad2deg(grid_data["grid_det"]["clon"]) <= (x0+wx)) & 
                          (np.rad2deg(grid_data["grid_det"]["clat"]) >= y0) & (np.rad2deg(grid_data["grid_det"]["clat"]) <= (y0+wy))).values

grid_data["cells_ens"] = ((np.rad2deg(grid_data["grid_ens"]["clon"]) >= x0) & (np.rad2deg(grid_data["grid_ens"]["clon"]) <= (x0+wx)) & 
                          (np.rad2deg(grid_data["grid_ens"]["clat"]) >= y0) & (np.rad2deg(grid_data["grid_ens"]["clat"]) <= (y0+wy))).values

In [ ]:
# Plotting setup:
lvls = [0,0.1,1,2,5,10,15,20,30,50,75,100,125,150,200]
norm = colors.BoundaryNorm(lvls, 256)

# Maps

In [ ]:
for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/CTRL/{datetime}/tot_prec_det.nc")["TOT_PREC"]

    fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')

    im = ax.tricontourf(tri, ds.sel(time="2021-07-15T00") - ds.sel(time="2021-07-13T00"), levels=lvls, norm=norm)

    # Create a Rectangle patch
    rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    plt.title(f"DET CTRL - Init: {datetime}")
    plt.colorbar(im, label="48h total Precipitation in kg / m^2", ticks=lvls)
    plt.savefig(f'./figs/ctrl_det_{datetime}_48h.png', dpi=300, bbox_inches='tight', format='png')
    plt.show()

In [ ]:
for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/SATU/{datetime}/tot_prec_det.nc")["tot_prec"]

    fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')

    im = ax.tricontourf(tri, ds.sel(time="2021-07-15T00") - ds.sel(time="2021-07-13T00"), levels=lvls, norm=norm)

    # Create a Rectangle patch
    rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    plt.title(f"DET SATU - Init: {datetime}")
    plt.colorbar(im, label="48h total Precipitation in kg / m^2", ticks=lvls)
    plt.savefig(f'./figs/satu_det_{datetime}_48h.png', dpi=300, bbox_inches='tight', format='png')
    plt.show()

In [ ]:
for datetime in ini_dates:
    ds = xr.open_dataset(f"./data/WILT/{datetime}/TOT_PREC_det.nc")["TOT_PREC"]

    fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')

    im = ax.tricontourf(tri, ds.sel(time="2021-07-15T00") - ds.sel(time="2021-07-13T00"), levels=lvls, norm=norm)

    # Create a Rectangle patch
    rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    plt.title(f"DET SATU - Init: {datetime}")
    plt.colorbar(im, label="48h total Precipitation in kg / m^2", ticks=lvls)
    plt.savefig(f'./figs/wilt_det_{datetime}_48h.png', dpi=300, bbox_inches='tight', format='png')
    plt.show()

# Time Series

In [ ]:
def hourly_precip(ds, grid_data, run_type):
    """
    Disaggregates the cumulative precipitation data into an hourly time series.
    """
    area_prec = np.full(len(ds), np.nan)
    area_prec[0] = 0.

    for i in range(1,len(area_prec)):
        area_prec[i] = ((ds.isel(time=i) - ds.isel(time=i-1)) * 
                        grid_data[f"area_{run_type}"].data).isel(ncells=grid_data[f"cells_{run_type}"]).sum().item()
    
    return(area_prec)

In [ ]:
run_names = ["det"] + [f"mem{i:03d}" for i in range(1,21)]

In [ ]:
for datetime in ini_dates[1:]:
    datetime_str = f"{str(datetime)[0:4]}-{str(datetime)[4:6]}-{str(datetime)[6:8]}T12"

    # Control run:
    dts_ctrl = {}
    area_precs_ctrl = {}

    for mem in run_names:
        ds = xr.open_dataset(f"./data/CTRL/{datetime}/TOT_PREC_{mem}.nc")["TOT_PREC"]
        dts_ctrl[mem] = ds["time"]

        if mem == "det":
            area_precs_ctrl[mem] = hourly_precip(ds, grid_data, "det")
        else:
            area_precs_ctrl[mem] = hourly_precip(ds, grid_data, "ens")

    ens_mean_ctrl = np.array([area_precs_ctrl[mem] for mem in run_names[1:]]).mean(axis=0)
    ens_std_ctrl = np.array([area_precs_ctrl[mem] for mem in run_names[1:]]).std(axis=0)


    # Saturated scenario:
    dts_satu = {}
    area_precs_satu = {}

    for mem in run_names:
        ds = xr.open_dataset(f"./data/SATU/{datetime}/TOT_PREC_{mem}.nc")["TOT_PREC"]
        dts_satu[mem] = ds["time"]

        if mem == "det":
            area_precs_satu[mem] = hourly_precip(ds, grid_data, "det")
        else:
            area_precs_satu[mem] = hourly_precip(ds, grid_data, "ens")

    ens_mean_satu = np.array([area_precs_satu[mem] for mem in run_names[1:]]).mean(axis=0)
    ens_std_satu = np.array([area_precs_satu[mem] for mem in run_names[1:]]).std(axis=0)


    # Wilting point scenario:
    dts_wilt = {}
    area_precs_wilt = {}

    for mem in run_names:
        ds = xr.open_dataset(f"./data/WILT/{datetime}/TOT_PREC_{mem}.nc")["TOT_PREC"]
        dts_wilt[mem] = ds["time"]

        if mem == "det":
            area_precs_wilt[mem] = hourly_precip(ds, grid_data, "det")
        else:
            area_precs_wilt[mem] = hourly_precip(ds, grid_data, "ens")

    ens_mean_wilt= np.array([area_precs_wilt[mem] for mem in run_names[1:]]).mean(axis=0)
    ens_std_wilt = np.array([area_precs_wilt[mem] for mem in run_names[1:]]).std(axis=0)

    
    # Plot:
    fig, ax = plt.subplots()

    line = ax.plot(dts_ctrl["det"], area_precs_ctrl["det"], label="CTL")
    ax.fill_between(dts_ctrl["mem001"], ens_mean_ctrl + ens_std_ctrl, ens_mean_ctrl - ens_std_ctrl, color=line[0].get_color(), alpha=0.5)

    line = ax.plot(dts_satu["det"], area_precs_satu["det"], label="SAT")
    ax.fill_between(dts_satu["mem001"], ens_mean_satu + ens_std_satu, ens_mean_satu - ens_std_satu, color=line[0].get_color(), alpha=0.5)

    line = ax.plot(dts_wilt["det"], area_precs_wilt["det"], label="WLT")
    ax.fill_between(dts_wilt["mem001"], ens_mean_wilt + ens_std_wilt, ens_mean_wilt - ens_std_wilt, color=line[0].get_color(), alpha=0.5)

    ax.set(title=f"Comparison for Initialization at {datetime_str}", ylabel="Total Area Precipitation in mm/h")
    ax.legend()

    plt.xticks(rotation=30)
    plt.xlim(np.datetime64(datetime_str), np.datetime64(datetime_str) + np.timedelta64(5,"D"))

    plt.show()